# Video Tracking — Bikes

## Imports

In [ ]:
import holoviews as hv
from hvplot import pandas
from holoviews import opts

import sys
sys.path.append("../src")

from mobiml.datasets import CopenhagenCyclists
from bokeh.io import output_notebook

output_notebook()

opts.defaults(opts.Overlay(active_tools=['wheel_zoom']))

## Load bike data

In [ ]:
bikes = CopenhagenCyclists(r"../tests/data/test_bike.pickle")

## Plot with intersection background

In [ ]:
bg_img = hv.RGB.load_image(r"../examples/data/intersection2.png", bounds=(0,0,640,360)) 
bg_img * bikes.to_df().hvplot.scatter(
    x="x", y="y", width=900, height=500, by="traj_id", 
    alpha=0.5, hover_cols=["traj_id", "frame", "x", "y"])

## Area Aggregation

In [ ]:
import geopandas as gpd
from shapely.geometry import box
from mobiml.transforms import AreaAggregator
from mobiml.preprocessing import TrajectoryEnricher

areas = gpd.GeoDataFrame({
    "area": [1, 2, 3],
    "geometry": [
        box(0, 250, 630, 350),   # area 1: top 
        box(0, 0, 320, 250),     # area 2: left
        box(320, 0, 630, 250),   # area 3: right
    ]
})

bikes_agg = bikes.copy()
bikes_agg = TrajectoryEnricher(bikes_agg).add_features(speed=True, direction=True)
bikes_agg.running_number_added=False

area_stats = AreaAggregator(bikes_agg).aggregate(areas)
area_stats[["area", "point_count", "point_density", "avg_speed", "avg_direction"]]

In [ ]:
bg_img = bg_img * area_stats.hvplot(
    c="point_count", alpha=0.4, width=900, height=500,
    colorbar=True, line_color="white", line_width=2,
    hover_cols=["area", "point_count", "point_density", "avg_speed", "avg_direction"]
)
bg_img * bikes.to_df().hvplot.scatter(
    x="x", y="y", width=900, height=500, by="traj_id", alpha=0.5,)